[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_6_Mars_Rover_Capstone_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


<div dir="rtl" style="text-align:right">

# 🚀 اللاب 6: المشروع النهائي الشامل — الفروق المحدودة، خوارزمية SOR، والملاحة الذاتية على المريخ
### *غرفة التحكم: دمج المعادلات التفاضلية الجزئية في محرك ملاحة المركبة الكامل*
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)  
**المرجع:** التكامل التطبيقي الشامل لما درسناه في اللابات 1–5 مع الكود الفعلي في [`rover_path_finder/solvers.py`](rover_path_finder/solvers.py) و [`rover_path_finder/planners.py`](rover_path_finder/planners.py)

---
### 🗺️ خريطة الكورس الكاملة: كيف تجمّعت قطع الليغو الخمس (اللابات 1–5) لبناء محرك الملاحة؟
```
[اللاب 1: حساب الميول slope_map] ──► [اللاب 2: خطوط لاجرانج descend] ──► [اللاب 3: مسافة الأمان dilate]
                                                                                      │
📍 [اللاب 6: الحل العددي SOR + Neumann وإطلاق المركبة] ◄── [اللاب 5: مشكلة الممرات الضيقة] ◄── [اللاب 4: معادلة لابلاس _avg]
```

| قطعة البناء (اللاب) | القانون الرياضي في مذكرة الـ PDF (ص 1–62) | المهمة الهندسية التي يحلها داخل مركبة المريخ | الدالة الفعلية في المشروع (`rover_path_finder/`) |
| :--- | :--- | :--- | :--- |
| **🧱 القطعة 1 (اللاب 1)** | $p = \frac{\partial z}{\partial x},\; q = \frac{\partial z}{\partial y}$ وحذف الثوابت الاختيارية *(PDF ص 2–11)* | قياس ميل التضاريس $\|\nabla z\| = \sqrt{p^2+q^2}$ واكتشاف المنحدرات الخطرة (`> 0.30`) | [`terrain.slope_map(elev)`](rover_path_finder/terrain.py) |
| **🧱 القطعة 2 (اللاب 3)** | المعادلات غير الخطية ومعادلة المسافة $p^2 + q^2 = 1$ *(PDF ص 17–27)* | توسيع كل صخرة وحفرة بمقدار نصف قطر جسم المركبة $R$ لمنع اصطدام العجلات | [`terrain.dilate(raw, rover_radius)`](rover_path_finder/terrain.py) |
| **🧱 القطعة 3 (اللاب 4)** | معادلة لابلاس $(D_x^2 + D_y^2)\phi = 0$ والجذور المركبة $m = \pm i$ *(PDF ص 28–51)* | بناء سطح جهد توافقي أملس يستحيل رياضياً أن يحتوي على أي قاع محلي يحبس المركبة | [`solvers._avg(phi)`](rover_path_finder/solvers.py) |
| **🧱 القطعة 4 (اللاب 5)** | العامل الخطي $(aD_x+bD_y+c)z=0 \implies e^{-(c/a)x}$ *(PDF ص 52–62)* | تفسير تلاشي الجهد $e^{-\pi x/w}$ في الممرات الضيقة واستبدال ديريكليه بشرط نيومان العازل | [`planners.neumann_potential()`](rover_path_finder/planners.py) |
| **🧱 القطعة 5 (اللاب 2)** | معادلة لاجرانج للخصائص $\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$ *(PDF ص 12–17)* | تتبع خط الانسياب الأشد انحداراً $(-\phi_x, -\phi_y)$ لقيادة المركبة من البداية للهدف | [`planners.descend(phi, hazard, start, goal)`](rover_path_finder/planners.py) |

---
### 🧭 حجر رشيد (The Rosetta Stone): الربط المباشر بين المعادلات وملفات المشروع

| 1. المفهوم الرياضي | 2. المعنى الفيزيائي والهندسي | 3. الكود الفعلي في `rover_path_finder/` |
| :--- | :--- | :--- |
| $\nabla^2\phi = \frac{\partial^2\phi}{\partial x^2} + \frac{\partial^2\phi}{\partial y^2} = 0$ | **معادلة لابلاس (من اللاب 4):** حقل جهد توافقي خالٍ تماماً من القيعان المحلية | المعادلة الحاكمة في [`solvers.py`](rover_path_finder/solvers.py) |
| $\phi_{i,j} = \frac{1}{4}(\phi_{i+1,j} + \phi_{i-1,j} + \phi_{i,j+1} + \phi_{i,j-1})$ | **قالب النقاط الخمس (5-Point Stencil):** تقريب المشتقات الثانية على شبكة بكسل | الدالة [`_avg(phi)`](rover_path_finder/solvers.py) في `solvers.py` |
| $\frac{\partial\phi}{\partial n} = 0$ على حدود العوائق | **شرط نيومان العازل (من اللاب 5):** الجهد ينساب حول الصخور ولا يتلاشى في الممرات الضيقة | قسمة مجموع الجيران الحرة على عددها الفعلي `cnt` في [`sor_neumann`](rover_path_finder/solvers.py) |
| $\omega_{\text{opt}} = \frac{2}{1 + \sin(\pi/N)}$ | **معامل الاسترخاء الفائق المتتالي (SOR):** تسريع تقارب الشبكة عشرات المرات | الدالة [`optimal_omega(N)`](rover_path_finder/solvers.py) في `solvers.py` |
| $\frac{dx}{-\phi_x} = \frac{dy}{-\phi_y}$ | **معادلة لاجرانج لخطوط الانسياب (من اللاب 2):** تتبع المسار من البداية للهدف | الدالة [`descend(phi, hazard, start, goal)`](rover_path_finder/planners.py) |

---
## 1. لماذا ننتقل من الحل التحليلي بالورقة والقلم إلى الحل العددي على الشبكة؟

في اللابات 1 إلى 5، حللنا المعادلات التفاضلية الجزئية تحليلياً بالورقة والقلم على نطاقات هندسية بسيطة.
ولكن في مشروع مركبة المريخ لدينا:
* بيئة الاختبار القياسية `make_terrain(N=100)` تتكون من شبكة **$100 \times 100 = 10,000$ عقدة**، وتضاريس وادي كاسي المريخية الحقيقية (`kasei.IMG` في `main.py`) تصل إلى **$160 \times 160 = 25,600$ عقدة** مع آلاف الصخور والحفر غير المنتظمة!
* يستحيل كتابة دالة تحليلية مغلقة واحدة تحقق شرط العزل $\frac{\partial\phi}{\partial n} = 0$ على حدود آلاف الصخور المتعرجة.
* لذلك نحول معادلة لابلاس المستمرة $\nabla^2\phi = 0$ إلى نظام معادلات جبرية على الشبكة باستخدام **طريقة الفروق المحدودة (Finite Difference Method)** ونحله تكرارياً بسرعة فائقة باستخدام **خوارزمية Red-Black SOR**.

> 📌 **ملاحظة منهجية صادقة:** دراسة معادلة لابلاس التحليلية $(D_x^2 + D_y^2)z = 0$ موجودة في المذكرة (ص 39 / PDF ص 33)، أما **تقطيع الشبكة (5-Point Stencil) ومبدأ القيمة العظمى (Maximum Principle) وخوارزمية SOR** فهي الإثراء العددي والهندسي الذي يربط مذكرة المحاضرات بمحرك البرمجيات الحقيقي في [`rover_path_finder/solvers.py`](rover_path_finder/solvers.py).

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab6

%matplotlib inline
print("تم تهيئة اللاب 6 النهائي بنجاح!")


<div dir="rtl" style="text-align:right">

---
## 2. الاشتقاق الرياضي خطوة بخطوة: من متسلسلة تايلور إلى برهان استحالة المصائد المحلية

### أولاً: اشتقاق قالب النقاط الخمس (5-Point Stencil) من متسلسلة تايلور
عند أي عقدة داخلية $(i, j)$ تبعد مسافة $h$ عن جيرانها الأربعة:
$$\phi_{i+1, j} = \phi_{i,j} + h \left.\frac{\partial\phi}{\partial x}\right|_{i,j} + \frac{h^2}{2}\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} + \frac{h^3}{6}\left.\frac{\partial^3\phi}{\partial x^3}\right|_{i,j} + \mathcal{O}(h^4)$$
$$\phi_{i-1, j} = \phi_{i,j} - h \left.\frac{\partial\phi}{\partial x}\right|_{i,j} + \frac{h^2}{2}\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} - \frac{h^3}{6}\left.\frac{\partial^3\phi}{\partial x^3}\right|_{i,j} + \mathcal{O}(h^4)$$

بجمع المعادلتين معاً، تختفي المشتقات الفردية ($\partial\phi/\partial x$ و $\partial^3\phi/\partial x^3$) تماماً:
$$\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} \approx \frac{\phi_{i+1, j} - 2\phi_{i,j} + \phi_{i-1, j}}{h^2}, \qquad \left.\frac{\partial^2\phi}{\partial y^2}\right|_{i,j} \approx \frac{\phi_{i, j+1} - 2\phi_{i,j} + \phi_{i, j-1}}{h^2}$$

بالتعويض في معادلة لابلاس $\frac{\partial^2\phi}{\partial x^2} + \frac{\partial^2\phi}{\partial y^2} = 0$ والضرب في $h^2$:
$$(\phi_{i+1, j} + \phi_{i-1, j} + \phi_{i, j+1} + \phi_{i, j-1}) - 4\phi_{i,j} = 0 \implies \boxed{\phi_{i,j} = \frac{1}{4}\left(\phi_{i+1, j} + \phi_{i-1, j} + \phi_{i, j+1} + \phi_{i, j-1}\right)}$$

---
### ثانياً: البرهان الرياضي البسيط لاستحالة وجود أي مصيدة محلية (Discrete Maximum Principle)
لماذا نضمن بنسبة 100% أن المركبة لن تعلق أبداً في أي حفرة جهد وهمية (Local Minimum)؟
* **البرهان في سطر واحد:** لنفترض جدلاً أن عقدة داخلية حرة $(i,j)$ هي "قاع محلي صارم" (Strict Local Minimum)، أي أن قيمتها أصغر من جميع جيرانها الأربعة:
  $$\phi_{i,j} < \phi_{\text{North}}, \quad \phi_{i,j} < \phi_{\text{South}}, \quad \phi_{i,j} < \phi_{\text{East}}, \quad \phi_{i,j} < \phi_{\text{West}}$$
* بجمع المتباينات الأربع والقسمة على 4 نحصل على:
  $$\phi_{i,j} < \frac{1}{4}\left(\phi_{\text{North}} + \phi_{\text{South}} + \phi_{\text{East}} + \phi_{\text{West}}\right)$$
* ولكن معادلة لابلاس تفرض المساواة التامة $\phi_{i,j} = \frac{1}{4}(\phi_N + \phi_S + \phi_E + \phi_W)$! **تناقض رياضي مستحيل!**
* إذن: **لا يمكن لأي عقدة داخلية حرة أن تكون قاعاً محلياً**؛ القاع الوحيد على الخريطة بأكملها هو نقطة الهدف نفسها ($\phi = 0$)!

---
### ثالثاً: كيف نطبق شرط نيومان العازل ($\frac{\partial\phi}{\partial n} = 0$) على جدران الحفر؟
إذا كان أحد الجيران الأربعة صخرة أو عائقاً (`free == False`)، فلا يجب أن نأخذ قيمته في المتوسط! بدلاً من القسمة على 4 دائماً، نجمع **فقط الجيران الحرة المتاحة** (`pf = phi * f`) ونقسم على **عدد الجيران الحرة الفعلي** `cnt` (والذي قد يكون 3 بمحاذاة جدار مستقيم، أو 2 في زاوية):
$$\phi_{i,j} = \frac{\sum_{\text{free neighbors}} \phi_{\text{neighbor}}}{\text{cnt}_{i,j}}$$
وهذا يكافئ رياضياً جعل المشتقة العمودية على الجدار تساوي صفراً ($\frac{\partial\phi}{\partial n} = 0$).

</div>


<div dir="rtl" style="text-align:right">

---
## 3. 📝 تحدي الورقة والقلم 1 (الحساب اليدوي للتكرار الأول بطريقة جاكوبي — Jacobi Sweep 1)

لكي تشعر بما يفعله الكود في كل تكرار، لنحسب يدوياً **المسحة الأولى بطريقة جاكوبي المتزامنة (Jacobi Sweep 1)** على شبكة صغيرة $4 \times 4$ تحتوي على 4 خلايا داخلية $(A, B, C, D)$ مهيأة في البداية بالقيمة $\phi^{(0)} = 0.0$:

```
            الحد العلوي (ثابت: phi = 1.0)
         +--------+--------+--------+--------+
         |  1.0   |  1.0   |  1.0   |  1.0   |
         +--------+--------+--------+--------+
الحد     |  0.5   | A(0.0) | B(0.0) |  0.5   | الحد الأيمن
الأيسر   +--------+--------+--------+--------+ (ثابت Dirichlet:
(ثابت:   |  0.5   | C(0.0) | D(0.0) |  0.5   |  phi = 0.5)
phi=0.5) +--------+--------+--------+--------+
         |  0.0   |  0.0   |  0.0   |  0.0   |
         +--------+--------+--------+--------+
            الحد السفلي (ثابت: phi = 0.0)
```

> 📌 **ملاحظة مهمة:** في طريقة **جاكوبي (Jacobi)**، نحسب جميع القيم الجديدة $\phi^{(1)}$ باستخدام القيم القديمة $\phi^{(0)} = 0.0$ فقط:
> * جيران الخلية $A$ في البداية هم: أعلى $= 1.0$، أسفل ($C^{(0)}$) $= 0.0$، يسار $= 0.5$، يمين ($B^{(0)}$) $= 0.0$.
> * جيران الخلية $B$ في البداية هم: أعلى $= 1.0$، أسفل ($D^{(0)}$) $= 0.0$، يسار ($A^{(0)}$) $= 0.0$، يمين $= 0.5$.
> *(معلومة إضافية: لو كنا نستخدم طريقة Gauss-Seidel المتسلسلة، لكنا استخدمنا قيمة $A^{(1)} = 0.375$ الجديدة فوراً عند حساب $B^{(1)}$ فتعطي $0.46875$؛ لكن هنا نطلب تحديث **جاكوبي المتزامن**).*

احسب $\phi_A^{(1)}$ و $\phi_B^{(1)}$ بطريقة جاكوبي وأدخلهما أدناه:

<details>
<summary>💡 <b>تلميح (اضغط هنا إذا احتجت مساعدة)</b></summary>

اجمع الجيران الأربعة الابتدائيين للخلية $A$: $(1.0 + 0.0 + 0.5 + 0.0) / 4 = 1.5 / 4$. وبسبب التماثل الأفقي للشبكة، فإن الخلية $B$ لها نفس مجموع الجيران الابتدائيين تماماً!
</details>

</div>


In [ ]:
# خلية تقييم: حساب الشبكة اليدوي (Jacobi Sweep 1)
### START CODE HERE ###
phi_A = 0.375
phi_B = 0.375
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab6.check_paper_challenge_1(phi_A, phi_B)


<div dir="rtl" style="text-align:right">

---
## 4. 🚀 تحدي الورقة والقلم 2 (معامل الاسترخاء الأمثل $\omega_{\text{opt}}$ — الدالة `optimal_omega`)

طريقة جاكوبي بطيئة جداً على الشبكات الكبيرة لأن المعلومات تنتقل بمعدل خلية واحدة في كل تكرار.
لتسريع الحل عشرات المرات، نستخدم **الاسترخاء الفائق المتتالي (Successive Over-Relaxation - SOR)**، الذي يأخذ خطوة جريئة في اتجاه التغيير بمعامل تضخيم $\omega \in (1, 2)$:
$$\phi_{i,j}^{\text{new}} = (1 - \omega)\,\phi_{i,j}^{\text{old}} + \omega\,\text{avg}_{i,j}$$

وفقاً لنظرية يونغ (Young's Theorem) المطبقة في دالة [`optimal_omega(N)`](rover_path_finder/solvers.py) في ملف `solvers.py`، يُعطى المعامل الأمثل لشبكة حجمها $N$ بالعلاقة:
$$\omega_{\text{opt}} = \frac{2}{1 + \sin\left(\frac{\pi}{N}\right)}$$

احسب قيمة $\omega_{\text{opt}}$ عندما يكون $N = 16$:

<details>
<summary>💡 <b>تلميح (اضغط هنا إذا احتجت مساعدة)</b></summary>

عوض بـ $N = 16$ في القانون `2.0 / (1.0 + np.sin(np.pi / 16))` (أو احسبها بالآلة الحاسبة بوضع الراديان: $\sin(\pi/16) \approx 0.19509 \implies \omega_{\text{opt}} \approx 1.6735$).
</details>

</div>


In [ ]:
# خلية تقييم: معامل أوميجا الأمثل لـ N=16
### START CODE HERE ###
omega_16 = 2.0 / (1.0 + np.sin(np.pi / 16))
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab6.check_paper_challenge_2(omega_16)


<div dir="rtl" style="text-align:right">

---
## 5. 💻 التطبيق البرمجي ببايثون: بناء محرك `sor_neumann_step` الفعلي

الآن ستكتب بنفسك القلب النابض لملف [`rover_path_finder/solvers.py`](rover_path_finder/solvers.py): دالة `sor_neumann_step` التي تجمع بين:
1. **تقسيم رقعة الشطرنج (Red-Black Ordering):** تحديث الخلايا الحمراء (`(x + y) % 2 == 0`) دفعة واحدة باستخدام متجهات NumPy السريعة، ثم تحديث الخلايا السوداء (`(x + y) % 2 == 1`) بالاعتماد على القيم الحمراء المحدثة للتو!
2. **شرط نيومان العازل (`pf = phi * f` والقسمة على `cnt`):** لضمان عدم تسرب الجهد داخل العوائق.
3. **تحديث الاسترخاء الفائق (SOR):** `phi[mk] = (1 - omega) * old + omega * avg[mk]`.

</div>


In [ ]:
# دالة تقييم برمجية: sor_neumann_step
def sor_neumann_step(phi, fixed, free, omega=1.5, tol=1e-5, maxit=5000):
    # خوارزمية Red-Black SOR مع جدران نيومان العازلة للعوائق (مطابقة لـ solvers.py)
    phi = phi.copy()
    n, m = phi.shape
    f = free.astype(float)
    
    # حساب عدد الجيران الحرة (من 1 إلى 4) لكل خلية لتطبيق شرط نيومان d(phi)/dn = 0
    cnt = sum(np.roll(f, s, a) for a in (0, 1) for s in (1, -1))
    cnt = np.maximum(cnt, 1.0)
    
    yy, xx = np.mgrid[0:n, 0:m]
    parity = (xx + yy) % 2
    masks = [(parity == c) & free & ~fixed for c in (0, 1)]
    phi[~free] = 0.0
    hist = []
    
    ### START CODE HERE ###
    for _ in range(maxit):
        d = 0.0
        for mk in masks:
            pf = phi * f
            avg = (
                np.roll(pf, 1, 0) + np.roll(pf, -1, 0)
                + np.roll(pf, 1, 1) + np.roll(pf, -1, 1)
            ) / cnt
            old = phi[mk]
            phi[mk] = (1 - omega) * old + omega * avg[mk]
            d = max(d, float(np.abs(phi[mk] - old).max()))
        hist.append(d)
        if d < tol:
            break
    ### END CODE HERE ###
    
    return phi, hist

# تشغيل الاختبار الآلي:
test_lab6.test_sor_neumann_step(sor_neumann_step)


<div dir="rtl" style="text-align:right">

---
## 6. 🌉 لماذا تتفوق معادلة لابلاس على طريقة المجالات الاصطناعية الكلاسيكية (APF)؟

في الروبوتات التقليدية، تستخدم طريقة **مجال الجهد الاصطناعي (Artificial Potential Field - APF)** الموجودة في [`planners.apf_potential`](rover_path_finder/planners.py) جمعاً جبرياً بسيطاً لجهد جاذب نحو الهدف وجهد طارد من الصخور:
$$U(\mathbf{x}) = U_{\text{att}}(\mathbf{x}) + U_{\text{rep}}(\mathbf{x})$$

**أين تكمن الكارثة الهندسية في APF؟**
* عندما تواجه المركبة عائقاً مقعراً أو حفرة على شكل حرف **U**، تسحب قوة الجذب المركبة للأمام نحو الهدف خلف الجدار، بينما تدفعها قوة التنافر للخلف بعيداً عن الجدار.
* عند نقطة داخل الحفرة، **تتساوى القوتان في المقدار وتتعاكسان في الاتجاه** ($\nabla U = \mathbf{0}$)، فتتكون **مصيدة قاع محلي (Local Minimum Trap)** لأن $\nabla^2 U \neq 0$!
* أما في **معادلة لابلاس ($\nabla^2\phi = 0$)** مع شروط نيومان العازلة (`neumann_potential`)، فقد أثبتنا رياضياً بمبدأ القيمة العظمى استحالة وجود أي قاع محلي داخل المجال؛ لذلك يلتف تيار الجهد بسلاسة حول جدار الحفرة ويقود المركبة بأمان تام إلى الهدف!

---
## 7. 🎨 الإطلاق النهائي للمهمة: الملاحة الذاتية على تضاريس المريخ (`100x100`)

شغّل الخلية التالية لتوليد التضاريس المريخية وحل معادلة لابلاس واستخراج مسار المركبة (لاحظ كيف يتفادى المسار السماوي جميع مناطق الخطر الحمراء بفضل توافق الإحداثيات $(x, y)$ مع `descend`):

</div>


In [ ]:
from rover_path_finder.terrain import make_terrain
from rover_path_finder.planners import neumann_potential, descend, apf_potential, trap_map

# 1. توليد تضاريس مريخية 100x100 (10,000 عقدة) مع توسيع العوائق بنصف قطر المركبة
t_data = make_terrain(N=100, seed=42, rover_radius=3)
elev, raw_hazards, hazards = t_data["h"], t_data["raw"], t_data["hazard"]
start = t_data["start"]
goal = t_data["goal"]

# 2. حل معادلة لابلاس بشروط نيومان العازلة (tol=1e-6 لتنفيذ سريع ودقيق)
phi, sweeps, elapsed = neumann_potential(hazards, start=start, goal=goal, tol=1e-6)

# 3. تتبع مسار الانحدار مع خطوط الانسياب (معادلة لاجرانج من اللاب 2)
path, reached = descend(phi, hazards, start=start, goal=goal)

# 4. رسم حقل الجهد والمسار الفعلي (p[0] هو x=col، و p[1] هو y=row)
fig, ax = plt.subplots(figsize=(9.5, 7.5))
cs = ax.contourf(phi, levels=30, cmap='viridis')
plt.colorbar(cs, ax=ax, label=r"Harmonic Potential $\phi(x, y)$")
ax.contour(hazards, levels=[0.5], colors='red', linewidths=1.5)

px = [p[0] for p in path]
py = [p[1] for p in path]
ax.plot(px, py, color='cyan', linewidth=3, label="Rover Trajectory (Laplace Streamline)")
ax.plot(start[0], start[1], 'go', markersize=10, label=r"Start ($\phi=1$)")
ax.plot(goal[0], goal[1], 'r*', markersize=14, label=r"Goal ($\phi=0$)")

ax.set_title("Ares Rover Navigation: Laplace Streamlines Around Martian Hazards", fontsize=13)
ax.set_xlabel("x (column index)")
ax.set_ylabel("y (row index)")
ax.legend(loc='upper left')
plt.show()

print(f"✅ حالة المهمة: وصلت المركبة للهدف = {reached} | طول المسار = {len(path)} خطوة | عدد مسحات SOR = {sweeps} ({elapsed:.2f} ثانية)")


<div dir="rtl" style="text-align:right">

---
## 8. 🪤 الاختبار الحاسم: مقارنة مصيدة حرف U (APF مقابل معادلة لابلاس)

شغّل الخلية التالية لمشاهدة المقارنة المباشرة على خريطة المصيدة `trap_map(N=100)`:
* **على اليسار (APF):** تدخل المركبة من $(20, 50)$ باتجاه الهدف $(90, 50)$ فتعلق للأبد داخل جيب حرف U!
* **على اليمين (Laplace PDE):** يلتف المسار السماوي حول جدار المصيدة ويصل إلى الهدف $(90, 50)$ بنجاح تام!

</div>


In [ ]:
u_trap, trap_start, trap_goal = trap_map(N=100)

# 1. الروبوت التقليدي (APF)
apf_field = apf_potential(u_trap, goal=trap_goal)
apf_path, apf_reached = descend(apf_field, u_trap, start=trap_start, goal=trap_goal, maxsteps=200)

# 2. معادلة لابلاس التوافقية مع شروط نيومان (tol=1e-6)
laplace_field, trap_sweeps, trap_time = neumann_potential(u_trap, start=trap_start, goal=trap_goal, tol=1e-6)
laplace_path, laplace_reached = descend(laplace_field, u_trap, start=trap_start, goal=trap_goal, maxsteps=400)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# رسم الروبوت الكلاسيكي (APF)
axes[0].contourf(apf_field, levels=25, cmap='magma')
axes[0].contour(u_trap, levels=[0.5], colors='white', linewidths=1.5)
axes[0].plot([p[0] for p in apf_path], [p[1] for p in apf_path], 'r-', linewidth=3, label="APF Path (Stuck)")
axes[0].plot(trap_start[0], trap_start[1], 'go', markersize=9, label="Start (20, 50)")
axes[0].plot(trap_goal[0], trap_goal[1], 'y*', markersize=13, label="Goal (90, 50)")
axes[0].set_title("Classical Robotics (APF): TRAPPED in Local Minimum", fontsize=12)
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend(loc="upper left")

# رسم معادلة لابلاس (Laplace PDE)
axes[1].contourf(laplace_field, levels=25, cmap='viridis')
axes[1].contour(u_trap, levels=[0.5], colors='red', linewidths=1.5)
axes[1].plot([p[0] for p in laplace_path], [p[1] for p in laplace_path], 'c-', linewidth=3, label="Laplace Path (Escapes)")
axes[1].plot(trap_start[0], trap_start[1], 'go', markersize=9, label="Start (20, 50)")
axes[1].plot(trap_goal[0], trap_goal[1], 'y*', markersize=13, label="Goal (90, 50)")
axes[1].set_title("Laplace PDE: Maneuvers Cleanly Around the U-Trap", fontsize=12)
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

print("نتائج المقارنة الحاسمة:")
print(f"  • الروبوت التقليدي (APF) : هل وصل للهدف؟ {apf_reached} (علق داخل المصيدة المقعرة)")
print(f"  • معادلة لابلاس (Laplace) : هل وصل للهدف؟ {laplace_reached} (التف حول الجدار ووصل للهدف في {len(laplace_path)} خطوة و {trap_time:.2f} ثانية!)")


<div dir="rtl" style="text-align:right">

## 🏆 شهادة إتمام الكورس التطبيقي للمعادلات التفاضلية الجزئية

**تهانينا الحارة على إتمام اللابات الستة بنجاح!**

لقد بنيت بيدك الجسر الكامل بين **مذكرة المعادلات التفاضلية الجزئية** و**محرك ملاحة الفضاء `PDE-Rover-Engine`**:
1. **اللاب 1 (حذف الثوابت الاختيارية):** فهمت معنى المشتقتين الجزئيتين $(p, q)$ وربطتهما بحساب خريطة انحدار التضاريس [`terrain.slope_map`](rover_path_finder/terrain.py).
2. **اللاب 2 (طريقة لاجرانج $Pp+Qq=R$):** اشتققت المنحنيات المميزة وربطتها بمتتبع خطوط انسياب الحركة [`planners.descend`](rover_path_finder/planners.py).
3. **اللاب 3 (المعادلات غير الخطية وطريقة شاربيت):** حللت معادلة المسافة $p^2+q^2=1$ وربطتها بتوسيع العوائق بنصف قطر الأمان [`terrain.dilate`](rover_path_finder/terrain.py).
4. **اللاب 4 (المؤثرات المتجانسة من الرتب العليا):** حللت معادلة لابلاس $(D_x^2+D_y^2)z=0$ وفهمت لماذا تحقق دوالها التوافقية خاصية متوسط الجيران الأربعة [`solvers._avg`](rover_path_finder/solvers.py).
5. **اللاب 5 (المؤثرات غير المتجانسة في الرتبة):** حللت العوامل الخطية $(aD_x+bD_y+c)z=0$ وفهمت لماذا يتلاشى انحدار ديريكليه أسياً في الممرات الضيقة.
6. **اللاب 6 (الفروق المحدودة وخوارزمية SOR):** برمجت محرك `sor_neumann_step` مع شروط نيومان العازلة وأثبتَّ عملياً كيف تتغلب معادلة لابلاس على مصائد حرف U التي تعجز عنها خوارزميات الروبوتات التقليدية!

</div>
